# H2. The HSBC circuit on 129 qubits, and its exact noiseless features

HSBC/IBM (Ciceri et al., Sec. 3.2) use a Trotterized Heisenberg circuit on a 109-qubit chain of ibm_torino: a random
product state, then RZZ, RYY, RXX on neighboring pairs (odd bonds, then even bonds), with angles set by the features.
They read <X>, <Y>, <Z> on every qubit. Two settings: "shorter" (1 block, alpha = 1) and "longer" (2 blocks,
alpha = 0.1). Their reference is an approximate matrix-product-state simulation with an unreported bond dimension.

Here:
* the "longer" circuit (the one with their 34% gain) on a **129-qubit chain** of ibm_fez, the longest chain on the
  device; "shorter" is computed in simulation only;
* the **null circuit**: "longer" with every data angle set to zero. On hardware it runs the same physical gates, and
  its output cannot depend on the event;
* noiseless features by **Pauli propagation** with a stated truncation, checked against an exact statevector.

Run time: seconds (reads saved features). Recomputing takes about 15 minutes.

In [ ]:
RECOMPUTE = False
N_CHAIN   = 129
N_FEAT    = 216

In [ ]:
import os, sys, json, time, math, warnings
from pathlib import Path
# Find the study folder (the one holding the hsbcrep package), wherever Jupyter or VS Code started
def _find_root():
    starts = [Path(p).parent for p in [globals().get("__vsc_ipynb_file__")] if p]
    starts += [Path.cwd()]
    for s in starts:
        for p in [s, *s.parents]:
            if (p / "hsbcrep" / "__init__.py").exists():
                return p.resolve()
    raise RuntimeError(f"Cannot find the repository folder (the one holding hsbcrep/) from {Path.cwd()}. Set ROOT by hand in this cell.")
ROOT = _find_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repository folder:", ROOT)
OUT = ROOT / "outputs"; OUT.mkdir(exist_ok=True)
RAW = ROOT / "data" / "raw"
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
pd.set_option("display.width", 200)
import matplotlib.pyplot as plt
from hsbcrep import pqfm, cryptofill as CF
from hsbcrep.featuremap import cone_stats
from hsbcrep.lightcone import projections
from hsbcrep.pauliprop import projections_pp

## H2.1 The 1,000 events that go on hardware

125 per market day, drawn once with a fixed seed. All three arms (R, C, N) use these same events.

In [ ]:
ev = pd.read_csv(OUT / "events.csv"); F = pd.read_csv(OUT / "features.csv")
J, mu, sd = CF.hsbc_scale(F.values)                 # HSBC preprocessing: standardize, then 2 pi tanh(z / 3)
f_hw = OUT / "hw_events.npy"
if f_hw.exists() and not RECOMPUTE:
    hw = np.load(f_hw)
else:
    rng = np.random.default_rng(17)
    hw = np.sort(np.concatenate([rng.choice(np.flatnonzero(ev.day.values == d), 125, replace=False) for d in sorted(ev.day.unique())]))
    np.save(f_hw, hw)
np.save(OUT / "J_all.npy", J)
print(f"{len(hw)} events ({ev.y.values[hw].mean():.1%} filled), {len(np.unique(ev.day.values[hw]))} market days")

## H2.2 Circuits

In [ ]:
OPS = {"longer": pqfm.pqfm(N_CHAIN, N_FEAT, "longer"), "shorter": pqfm.pqfm(N_CHAIN, N_FEAT, "shorter"),
       "null": pqfm.pqfm(N_CHAIN, N_FEAT, "longer", null=True)}
rows = []
for name, ops in OPS.items():
    size, _, _ = cone_stats(N_CHAIN, ops)
    data = [abs(o.coeff) for o in ops if o.feats]
    rows.append({"circuit": name, "qubits": N_CHAIN, "two-qubit rotations": pqfm.n_data_gates(ops),
                 "CZ after decomposition": 2 * pqfm.n_data_gates(ops), "largest light cone": int(size.max()),
                 "max |angle| (rad)": round(max(data) * 2 * np.pi, 3) if data else 0.0})
pd.DataFrame(rows)

The "longer" circuit rotates by at most about 0.08 rad per gate. Its light cones reach 16 qubits.

## H2.3 Check: Pauli propagation against an exact statevector

In [ ]:
Pp, dr, _ = projections_pp(N_CHAIN, OPS["longer"], J[hw[:2]], tol=1e-6, qubits=[0, 64, 128])
Ps, _ = projections(N_CHAIN, OPS["longer"], J[hw[:2]], max_cone=17)
print(f"largest difference on 2 events x 3 qubits: {np.abs(Pp - Ps)[:, :, [0, 64, 128]].max():.1e}")

## H2.4 Noiseless features

In [ ]:
def compute(name, fn):
    f = OUT / f"noiseless_{name}.npy"
    if f.exists() and not RECOMPUTE:
        return np.load(f)
    t0 = time.time(); P = fn(); np.save(f, P); print(f"{name}: {time.time() - t0:.0f} s"); return P

NL = {"longer": compute("longer129_hw", lambda: projections_pp(N_CHAIN, OPS["longer"], J[hw], tol=1e-6)[0]),
      "shorter": compute("shorter129_hw", lambda: projections(N_CHAIN, OPS["shorter"], J[hw], max_cone=17)[0])}
null = compute("null129", lambda: projections(N_CHAIN, OPS["null"], J[hw[:1]], max_cone=17)[0][0])
for name, P in NL.items():
    Fl = P.reshape(len(P), -1)
    print(f"{name:8s}: feature spread across events, mean sd {Fl.std(0).mean():.4f} (max {Fl.std(0).max():.3f}); "
          f"mean Bloch length {np.sqrt((P ** 2).sum(1)).mean():.3f}")
print(f"null    : identical for every event; shot noise of one feature at 1,024 shots: {1 / np.sqrt(1024):.4f}")